In [ ]:
import pandas as pd

USE = "AUTO"  # or "CUSTOM"

if USE == "AUTO":
    try:
        # Automatically fetch S&P 500 tickers from Wikipedia
        table = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")[0]
        tickers = table["Symbol"].tolist()
        print(f"✅ Loaded {len(tickers)} tickers automatically.")
    except Exception as e:
        print("⚠️ Automatic ticker fetch failed, switching to manual list.")
        tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]
else:
    # Manual ticker list
    tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]


In [ ]:
# QuantLens - 01_data_fetching.ipynb
# Purpose: fetch OHLCV data for a chosen universe, save per-ticker CSVs in data/raw/
# Run this notebook from project root (QuantLens/)

import sys
import os
print("Python:", sys.version)
print("Working dir:", os.getcwd())


In [ ]:
# Imports
import pandas as pd
import numpy as np
import yfinance as yf
from tqdm.notebook import tqdm
import time
import os
import logging

# Paths
RAW_DIR = "data/raw/"
PROCESSED_DIR = "data/processed/"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Parameters
START_DATE = "2018-01-01"
END_DATE   = "2024-12-31"   # change as needed
PAUSE_BETWEEN = 0.3         # seconds between yfinance calls (reduce blocking)
MAX_RETRIES = 3


In [ ]:
# 3 options: 1) fetch S&P500 tickers from Wikipedia, 2) paste your own list, 3) use NIFTY50 list manually
USE = "SP500"   # options: "SP500", "NIFTY50", "CUSTOM"

def get_sp500_tickers():
    # fetch S&P500 tickers from Wikipedia table
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    tables = pd.read_html(url)
    df = tables[0]
    tickers = df['Symbol'].tolist()
    # yahoo uses '.' for some tickers? Usually fine for SP500
    return tickers

def get_nifty50_tickers():
    # you can paste a definitive list; here's a sample starter (top ones)
    return ["RELIANCE.NS","HDFCBANK.NS","TCS.NS","INFY.NS","ICICIBANK.NS",
            "SBIN.NS","HINDUNILVR.NS","KOTAKBANK.NS","LT.NS","AXISBANK.NS"]

if USE == "SP500":
    TICKERS = get_sp500_tickers()
elif USE == "NIFTY50":
    TICKERS = get_nifty50_tickers()
else:
    # Paste your custom list here
    TICKERS = ["AAPL","MSFT","GOOG"]  # replace with your tickers
len(TICKERS)


In [ ]:
def safe_download(ticker, start=START_DATE, end=END_DATE, max_retries=MAX_RETRIES):
    attempt = 0
    while attempt < max_retries:
        try:
            df = yf.download(ticker, start=start, end=end, progress=False, threads=False)
            if df is None or df.empty:
                raise ValueError("Empty dataframe")
            return df
        except Exception as e:
            attempt += 1
            wait = 1.0 * attempt
            print(f"[{ticker}] download failed attempt {attempt}/{max_retries}: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    print(f"[{ticker}] FAILED after {max_retries} attempts.")
    return None


In [ ]:
# IMPORTANT: If you have 500 tickers, do this in batches to avoid connection problems.
tickers_to_fetch = TICKERS  # or TICKERS[:50] for a test run
failures = []

for t in tqdm(tickers_to_fetch, desc="Downloading tickers"):
    out_file = os.path.join(RAW_DIR, f"{t}.csv")
    # skip if file exists (resume capability)
    if os.path.exists(out_file):
        continue
    df = safe_download(t)
    if df is None:
        failures.append(t)
        continue
    # yfinance returns index as DatetimeIndex; save with Date column
    df.index.name = "Date"
    df.reset_index(inplace=True)
    df.to_csv(out_file, index=False)
    time.sleep(PAUSE_BETWEEN)

print("Done. failures:", failures)


In [ ]:
import glob, random
files = glob.glob(RAW_DIR + "*.csv")
print("Files downloaded:", len(files))
if files:
    sample = random.choice(files)
    print("Sample file:", sample)
    df = pd.read_csv(sample, parse_dates=['Date'])
    display(df.head())


In [ ]:
def standardize_csv(infile, outfile=None):
    df = pd.read_csv(infile, parse_dates=['Date'])
    # ensure common columns exist
    cols = df.columns.str.lower()
    # yfinance names generally: Date, Open, High, Low, Close, Adj Close, Volume
    expected = ['Date','Open','High','Low','Close','Adj Close','Volume']
    # If Missing 'Adj Close' but has 'Close', set Adj Close = Close
    if 'Adj Close' not in df.columns and 'Adj Close' in df.columns.str.replace(' ',''):
        df.rename(columns={c: c.strip() for c in df.columns}, inplace=True)
    if 'Adj Close' not in df.columns:
        if 'Close' in df.columns:
            df['Adj Close'] = df['Close']
    # drop duplicated Date rows and sort
    df = df.drop_duplicates(subset=['Date']).sort_values('Date').reset_index(drop=True)
    if outfile:
        df.to_csv(outfile, index=False)
    return df

# apply to all raw files to rewrite cleaned versions (optional)
for f in files:
    standardize_csv(f, f)
print("Standardization done.")


In [ ]:
# Option A: Use yfinance.info (fast, but less complete)
def fetch_fundamentals_yf(ticker):
    t = yf.Ticker(ticker)
    info = t.info
    # pick specific fields you need
    fields = ['trailingPE','priceToBook','returnOnEquity','debtToEquity','marketCap']
    data = {k: info.get(k, None) for k in fields}
    data['ticker'] = ticker
    return data

# Example:
print(fetch_fundamentals_yf(TICKERS[0]))

# Option B: FinancialModelingPrep API (preferred for many fundamentals)
# You need an API key. If you have one, uncomment & use below:
"""
API_KEY = "YOUR_FMP_KEY"
import requests
def fetch_fmp_profile(ticker):
    url = f"https://financialmodelingprep.com/api/v3/profile/{ticker}?apikey={API_KEY}"
    r = requests.get(url)
    if r.ok:
        return r.json()
    return None
"""


In [ ]:
import glob
meta_rows = []
for f in glob.glob(RAW_DIR + "*.csv"):
    df = pd.read_csv(f, parse_dates=['Date'])
    ticker = os.path.basename(f).replace('.csv','')
    first = df['Date'].min()
    last = df['Date'].max()
    nrows = len(df)
    meta_rows.append({'ticker': ticker, 'start': first, 'end': last, 'nrows': nrows})

meta = pd.DataFrame(meta_rows).sort_values('ticker')
meta.to_csv("data/raw/manifest.csv", index=False)
display(meta.head(30))


In [ ]:
# create a "dev" sample of 20 tickers to speed up development and testing
sample_files = list(glob.glob(RAW_DIR + "*.csv"))[:20]
os.makedirs("data/dev/", exist_ok=True)
for f in sample_files:
    df = pd.read_csv(f)
    df.to_csv(os.path.join("data/dev/", os.path.basename(f)), index=False)
print("Sample dev dataset saved at data/dev/")


In [ ]:
print("Completed data fetching stage.")
print("Next: run `02_feature_engineering.ipynb` to compute factors (momentum, volatility, MA, returns).")
